In [1]:
import numpy as np
import matplotlib.pyplot as plt
import utils
import ot
import scipy
import os
import nbformat
import pandas as pd

In [2]:
nbformat.__version__

'5.10.4'

In [3]:
datasets_path = "datasets/action_smplx_models"

bools = []
for _ in range(1):
    while True:
        p = np.random.choice(os.listdir(datasets_path))
        if p != "male2_Calibration_stageii.npz":
            break

    p = datasets_path + "/" + p
    rand_idx = np.random.randint(0, int(np.load(p)["mocap_time_length"] * 120) - 120)
    td = 60

    points1, faces1 = utils.sampled_verts_from_path(p, idx = rand_idx, return_faces=True)
    points2, faces2 = utils.sampled_verts_from_path(p, idx = rand_idx + td, return_faces=True)
    mask = (points1[:,2] < np.percentile(points1[:,2], 10))
    utils.plot_arr(points1, color = ["red" if x else "blue" for x in mask])
    feet_only = points1[mask][:,:-1]
    # plt.scatter(feet_only[:,0], feet_only[:,1])
    # plt.scatter(points1.mean(axis = 0)[0], points1.mean(axis = 0)[1])
    diff = points2.mean(axis = 0) - points1.mean(axis=0)
    # plt.scatter(points1.mean(axis = 0)[0] + diff[0] / 25, points1.mean(axis = 0)[1] + diff[1] / 25)
    means = scipy.cluster.vq.kmeans(feet_only, 2)[0]
    #plt.scatter(kmeans[:,0], kmeans[:,1])
    middle_dists = (means - points1[:,:-1].mean(axis = 0)) @ (np.cross(np.append(diff[:-1], 0), np.array([0,0,1])))[:-1]
    l_ind = np.argmin(middle_dists)
    r_ind = np.argmax(middle_dists)
    m1dists = np.linalg.norm(points1 - np.array([means[r_ind, 0], means[r_ind, 1], 0]), axis = 1)
    m2dists = np.linalg.norm(points1 - np.array([means[l_ind, 0], means[l_ind, 1], 0]), axis = 1)
    fig = utils.plot_arr(np.append(points1, np.array([
        [means[0, 0], means[0, 1], 0],
        [means[1, 0], means[1, 1], 0]
    ]), axis = 0), color = ["blue" if x else "red" for x in (m1dists > m2dists)] + ["green"] * 2)

    bools.append("right" in utils.faces_to_regions([faces1[np.argmin(np.linalg.norm(points1 - np.array([means[r_ind, 0], means[r_ind, 1], 0]), axis = 1))]])[0])
    # red = right?
print(bools[0])
fig.show()

True


In [4]:
left_anchor_ind = np.argmin(np.linalg.norm(points1 - np.array([means[l_ind, 0], means[l_ind, 1], 0]), axis = 1))
right_anchor_ind = np.argmin(np.linalg.norm(points1 - np.array([means[r_ind, 0], means[r_ind, 1], 0]), axis = 1))

In [5]:
feet_l_ind = np.argmin(np.linalg.norm(feet_only - means[l_ind], axis = 1))
lai = np.argmin(np.linalg.norm(points1[:, :-1] - feet_only[feet_l_ind], axis = 1))

feet_r_ind = np.argmin(np.linalg.norm(feet_only - means[r_ind], axis = 1))
rai = np.argmin(np.linalg.norm(points1[:, :-1] - feet_only[feet_r_ind], axis = 1))

In [6]:
with_l_anchor = np.append(points1, points1[lai].reshape(-1, 3), axis = 0)
with_r_anchor = np.append(with_l_anchor, points1[rai].reshape(-1, 3), axis = 0)
utils.plot_arr(with_r_anchor, color = ["blue"] * 1000 + ["red"] * 2)

In [7]:
C = utils.graph_distance_within_cloud_minimally_connected(points1)

In [8]:
C = utils.graph_distance_within_cloud_minimally_connected(points1)
c = ["blue" if x else "red" for x in C[rai, :] > C[lai, :]]
c = C[rai, :] - C[lai, :]
utils.plot_arr(points1, color = c, colorbar = True)
# Blue = left, red = right

In [9]:
accs = []
betas = np.arange(0, 5, 0.1)
betas = [1]

a = np.ones(1000) / 1000
b = np.ones(1000) / 1000
M1 = ot.dist(points1, points2)
G1 = ot.solve(M1, a, b).plan

for beta in betas:
    augmented1 = utils.left_right_augmentation(points1, diff, beta = beta)
    augmented2 = utils.left_right_augmentation(points2, diff, beta = beta)

    M2 = ot.dist(augmented1, augmented2)

    G2 = ot.solve(M2, a, b).plan

    accs.append(utils.region_accuracy_adjusted(G2, faces1, faces2))



In [10]:
[i for i in range(5) if i != 2]

[0, 1, 3, 4]

In [11]:
utils.plot_3d_points_and_connections_region_matched(points1, points2, faces1, faces2, G1)

Region accuracy adjusted: 0.4700854700854701


In [12]:
utils.plot_3d_points_and_connections_region_matched(points1, points2, faces1, faces2, G2)

Region accuracy adjusted: 0.5448717948717948


In [13]:
utils.mismatch_counts(G1, faces1, faces2)

,mismatches
ordered,
"{'left_shin', 'right_shin'}",97
"{'left_thigh', 'right_thigh'}",58
"{'right_foot', 'left_foot'}",50
"{'right_thigh', 'right_hand'}",29
"{'left_hand', 'left_thigh'}",28
"{'upper_torso', 'left_upper_arm'}",26
"{'upper_torso', 'right_upper_arm'}",24
"{'lower_torso', 'right_forearm'}",24
"{'lower_torso', 'left_forearm'}",24


In [14]:
utils.mismatch_counts(G2, faces1, faces2)

,mismatches
ordered,
"{'lower_torso', 'pelvis'}",69
"{'left_thigh', 'pelvis'}",27
"{'upper_torso', 'left_upper_arm'}",25
"{'right_thigh', 'right_hand'}",25
"{'left_hand', 'left_thigh'}",24
"{'upper_torso', 'left_forearm'}",24
"{'upper_torso', 'lower_torso'}",23
"{'pelvis', 'right_forearm'}",21
"{'right_thigh', 'pelvis'}",20
